In [2]:
import gzip
import pandas as pd

LD_FILE   = "/gpfs/data/user/shreyags/TE_work/analysis/insertions_snps_LD_results/ld_results.ld"
ALU_VCF   = "/gpfs/data/user/shreyags/TE_work/filtered_vcfs/ALU.filtered.final.vcf.gz"
LINE1_VCF = "/gpfs/data/user/shreyags/TE_work/filtered_vcfs/LINE1.filtered.final.vcf.gz"
OUTDIR    = "/gpfs/data/user/shreyags/TE_work/replication_results/ld"

def load_vcf_positions(vcf_path):
    positions = set()
    with gzip.open(vcf_path, "rt") as f:
        for line in f:
            if line.startswith("#"):
                continue
            fields = line.split("\t")
            chrom  = fields[0].replace("chr", "")
            pos    = int(fields[1])
            positions.add((chrom, pos))
    print(f"  Loaded {len(positions):,} positions from {vcf_path}")
    return positions

print("Loading ALU positions...")
alu_pos = load_vcf_positions(ALU_VCF)

print("Loading LINE1 positions...")
line1_pos = load_vcf_positions(LINE1_VCF)

print("Loading LD file...")
ld = pd.read_csv(
    LD_FILE,
    sep=r"\s+",
    header=None,
    skiprows=1,
    names=["CHR_A", "BP_A", "SNP_A", "CHR_B", "BP_B", "SNP_B", "R2"]
)
print(f"  {len(ld):,} rows loaded")

ld["CHR_A"] = ld["CHR_A"].astype(str).str.replace("chr", "", regex=False)
ld["CHR_B"] = ld["CHR_B"].astype(str).str.replace("chr", "", regex=False)

def classify_row(row, alu_pos, line1_pos):
    for chrom_col, bp_col in [("CHR_A", "BP_A"), ("CHR_B", "BP_B")]:
        chrom = str(row[chrom_col])
        pos   = int(row[bp_col])
        if (chrom, pos) in alu_pos:
            return "ALU"
        if (chrom, pos) in line1_pos:
            return "LINE1"
    return None

print("Classifying rows...")
ld["MEI_TYPE"] = ld.apply(classify_row, axis=1, alu_pos=alu_pos, line1_pos=line1_pos)

print(f"\nClassification summary:")
print(ld["MEI_TYPE"].value_counts(dropna=False))

# ---------------------------------------------------
# split and save
# ---------------------------------------------------

alu_ld    = ld[ld["MEI_TYPE"] == "ALU"].drop(columns="MEI_TYPE")
line1_ld  = ld[ld["MEI_TYPE"] == "LINE1"].drop(columns="MEI_TYPE")
unmatched = ld[ld["MEI_TYPE"].isna()]

alu_ld.to_csv(f"{OUTDIR}/snp_ALU_ins.ld",    sep="\t", index=False)
line1_ld.to_csv(f"{OUTDIR}/snp_LINE1_ins.ld", sep="\t", index=False)

print(f"\nALU:      {len(alu_ld):,} rows -> {OUTDIR}/snp_ALU_ins.ld")
print(f"LINE1:    {len(line1_ld):,} rows -> {OUTDIR}/snp_LINE1_ins.ld")
print(f"Unmatched:{len(unmatched):,} rows")

if len(unmatched) > 0:
    print("\nSample unmatched rows:")
    print(unmatched.head())

Loading ALU positions...
  Loaded 26,977 positions from /gpfs/data/user/shreyags/TE_work/filtered_vcfs/ALU.filtered.final.vcf.gz
Loading LINE1 positions...
  Loaded 7,075 positions from /gpfs/data/user/shreyags/TE_work/filtered_vcfs/LINE1.filtered.final.vcf.gz
Loading LD file...
  183,555 rows loaded
Classifying rows...

Classification summary:
NaN      177456
ALU        5439
LINE1       660
Name: MEI_TYPE, dtype: int64

ALU:      5,439 rows -> /gpfs/data/user/shreyags/TE_work/replication_results/ld/snp_ALU_ins.ld
LINE1:    660 rows -> /gpfs/data/user/shreyags/TE_work/replication_results/ld/snp_LINE1_ins.ld
Unmatched:177,456 rows

Sample unmatched rows:
  CHR_A     BP_A           SNP_A CHR_B     BP_B           SNP_B        R2  \
0     1  4128797  chr1:4128797:A     1  4134718  chr1:4134718:C  0.948348   
1     1  4128797  chr1:4128797:A     1  4141438  chr1:4141438:A  0.933585   
2     1  4128994  chr1:4128994:A     1  4140016  chr1:4140016:T  0.941105   
3     1  4128994  chr1:4128994

In [3]:
import pandas as pd

alu_ins_path   = "/gpfs/data/user/shreyags/TE_work/replication_results/ld/snp_ALU_ins.ld"
line1_ins_path = "/gpfs/data/user/shreyags/TE_work/replication_results/ld/snp_LINE1_ins.ld"
gwas_path      = "/gpfs/data/user/shreyags/TE_work/data/gwas_data/gwas_associations_5_march2026.tsv"

alu_ins_ld   = pd.read_csv(alu_ins_path,   delim_whitespace=True, header=0)
line1_ins_ld = pd.read_csv(line1_ins_path, delim_whitespace=True, header=0)

print(f"ALU ins: {len(alu_ins_ld):,} rows  |  LINE1 ins: {len(line1_ins_ld):,} rows")

ALU ins: 5,439 rows  |  LINE1 ins: 660 rows


In [4]:
# ── Peek at the GWAS catalog ───────────────────────────────────────────────────
gwas_raw = pd.read_csv(gwas_path, sep="\t", low_memory=False, nrows=3)
print(gwas_raw.columns.tolist())

['DATE ADDED TO CATALOG', 'PUBMEDID', 'FIRST AUTHOR', 'DATE', 'JOURNAL', 'LINK', 'STUDY', 'DISEASE/TRAIT', 'INITIAL SAMPLE SIZE', 'REPLICATION SAMPLE SIZE', 'REGION', 'CHR_ID', 'CHR_POS', 'REPORTED GENE(S)', 'MAPPED_GENE', 'UPSTREAM_GENE_ID', 'DOWNSTREAM_GENE_ID', 'SNP_GENE_IDS', 'UPSTREAM_GENE_DISTANCE', 'DOWNSTREAM_GENE_DISTANCE', 'STRONGEST SNP-RISK ALLELE', 'SNPS', 'MERGED', 'SNP_ID_CURRENT', 'CONTEXT', 'INTERGENIC', 'RISK ALLELE FREQUENCY', 'P-VALUE', 'PVALUE_MLOG', 'P-VALUE (TEXT)', 'OR or BETA', '95% CI (TEXT)', 'PLATFORM [SNPS PASSING QC]', 'CNV']


In [5]:
gwas = pd.read_csv(gwas_path, sep="\t", low_memory=False)

gwas["SNPS"]     = gwas["SNPS"].astype(str).str.strip()
gwas["CHR_ID"]   = gwas["CHR_ID"].astype(str).str.strip()
gwas["CHR_POS"]  = pd.to_numeric(gwas["CHR_POS"], errors="coerce")
gwas["P-VALUE"]  = pd.to_numeric(gwas["P-VALUE"], errors="coerce")

print(f"GWAS catalog: {len(gwas):,} rows, {len(gwas.columns)} columns")

GWAS catalog: 1,064,375 rows, 34 columns


In [6]:
def clean_te_variant(id_series):
    cleaned = id_series.str.extract(r'^([^:]+:[^:]+)')[0]
    return cleaned.where(cleaned.str.startswith("chr"), "chr" + cleaned)

def add_chr_prefix(id_series):
    return id_series.where(id_series.str.startswith("chr"), "chr" + id_series)

def extract_ld_snps_insertion(df, insertion_type, te_tag):
    df.columns = ["chr_a", "bp_a", "id_a", "chr_b", "bp_b", "id_b", "r2"]
    is_te_in_a = df["id_a"].str.contains(te_tag, na=False)
    te_col    = df["id_a"].where(is_te_in_a, df["id_b"])
    proxy_col = df["id_b"].where(is_te_in_a, df["id_a"])
    return pd.DataFrame({
        "te_variant"   : clean_te_variant(te_col),
        "proxy_snp"    : add_chr_prefix(proxy_col),
        "r2"           : df["r2"],
        "variant_type" : insertion_type,
    })

alu_ins_tidy   = extract_ld_snps_insertion(alu_ins_ld,   "Alu",  "<INS:ME:ALU>")
line1_ins_tidy = extract_ld_snps_insertion(line1_ins_ld, "LINE1", "<INS:ME:LINE1>")

all_ld = pd.concat([alu_ins_tidy, line1_ins_tidy], ignore_index=True)

print(f"Total LD pairs: {len(all_ld):,}")
print(f"Unique proxy SNPs : {all_ld['proxy_snp'].nunique():,}")
print(f"Unique TE variants: {all_ld['te_variant'].nunique():,}")
print(all_ld.groupby('variant_type').size())
all_ld.head(3)

Total LD pairs: 6,099
Unique proxy SNPs : 5,921
Unique TE variants: 2,225
variant_type
Alu      5439
LINE1     660
dtype: int64


,te_variant,proxy_snp,r2,variant_type
0,chr1:4228405,chr1:4233940:G,0.737582,Alu
1,chr1:12006226,chr1:11993865:A,0.882614,Alu
2,chr1:15163638,chr1:15154634:C,0.867536,Alu


In [7]:
all_ld[["snp_chr", "snp_pos"]] = all_ld["proxy_snp"].str.extract(r"(?:chr)?(\w+):(\d+)")
all_ld["snp_pos"] = pd.to_numeric(all_ld["snp_pos"], errors="coerce")

print("Proxy SNP chr examples:", all_ld["snp_chr"].unique()[:5])
print("GWAS CHR_ID examples:  ", gwas["CHR_ID"].unique()[:5])

Proxy SNP chr examples: ['1' '2' '3' '4' '5']
GWAS CHR_ID examples:   ['17' '6' '7' '8' '1']


In [8]:
#Intersect with GWAS catalog on CHR + POS
hits = all_ld.merge(
    gwas,
    left_on=["snp_chr", "snp_pos"],
    right_on=["CHR_ID", "CHR_POS"],
    how="inner"
)

print(f"{len(hits):,} LD-proxy SNPs matched to GWAS associations")
print(f"  Unique proxy SNPs with GWAS hits : {hits['proxy_snp'].nunique():,}")
print(f"  Unique TE variants affected      : {hits['te_variant'].nunique():,}")
print(f"  Unique traits                    : {hits['DISEASE/TRAIT'].nunique():,}")
print(hits.groupby('variant_type').size())

508 LD-proxy SNPs matched to GWAS associations
  Unique proxy SNPs with GWAS hits : 163
  Unique TE variants affected      : 150
  Unique traits                    : 375
variant_type
Alu      347
LINE1    161
dtype: int64


In [10]:
#Reorder
priority_cols = [
    "variant_type", "te_variant", "proxy_snp", "SNPS", "r2",
    "MAPPED_GENE", "DISEASE/TRAIT", "P-VALUE", "OR or BETA", "95% CI (TEXT)",
]
remaining_cols = [c for c in hits.columns if c not in priority_cols]
hits_out = (hits[priority_cols + remaining_cols]
            .drop_duplicates()
            .loc[lambda d: d["OR or BETA"].astype(str).str.strip().replace({"nan": ""}) != ""]
            .sort_values(["variant_type", "te_variant", "P-VALUE"]))

base = "/gpfs/data/user/shreyags/TE_work/new_results/manuscript/supplementary_tables"
hits_out.to_csv(f"{base}/gwas_ld_hits_insertions.csv", index=False)
print(f"Insertions → {len(hits_out):,} rows saved to gwas_ld_hits_insertions.csv")
hits_out.head(10)

Insertions → 440 rows saved to gwas_ld_hits_insertions.csv


,variant_type,te_variant,proxy_snp,SNPS,r2,MAPPED_GENE,DISEASE/TRAIT,P-VALUE,OR or BETA,95% CI (TEXT),...,STRONGEST SNP-RISK ALLELE,MERGED,SNP_ID_CURRENT,CONTEXT,INTERGENIC,RISK ALLELE FREQUENCY,PVALUE_MLOG,P-VALUE (TEXT),PLATFORM [SNPS PASSING QC],CNV
196,Alu,chr10:3526833,chr10:3554334:G,rs7086377,0.944181,LINC02669 - KLF6,Waist-to-hip ratio adjusted for BMI,7.000000e-11,0.011900,[0.0084-0.0154] unit increase,...,rs7086377-T,0,7086377,intron_variant,1.0,0.3952,10.154902,NaN,NR [~ 27400000] (imputed),N
198,Alu,chr10:36304703,chr10:36319992:A,rs1775167,0.876804,LINC02630 - MTND5P17,Smoking initiation,2.000000e-26,0.008280,[0.0068-0.0098] unit increase,...,rs1775167-A,0,1775167,intergenic_variant,1.0,0.464,25.698970,NaN,NR [NR] (imputed),N
197,Alu,chr10:36304703,chr10:36319992:A,rs1775167,0.876804,LINC02630 - MTND5P17,Smoking initiation,3.000000e-25,0.009090,[0.0074-0.0108] unit increase,...,rs1775167-A,0,1775167,intergenic_variant,1.0,0.429,24.522879,NaN,NR [NR] (imputed),N
199,Alu,chr10:71223776,chr10:71224939:T,rs12355039,0.839465,UNC5B,Spelling,1.000000e-07,0.112000,[0.07-0.154] unit decrease,...,rs12355039-T,0,12355039,intron_variant,0.0,0.0859,7.000000,NaN,"Affymetrix, Illumina [7849740] (imputed)",N
211,Alu,chr11:100910662,chr11:100977284:T,rs7931273,0.878868,ARHGAP42,Mastocytosis,6.000000e-07,0.568100,[0.34-0.79],...,rs7931273-T,0,7931273,intron_variant,0.0,0.3236,6.221849,NaN,Illumina [281811],N
212,Alu,chr11:101041062,chr11:101097799:T,rs586143,0.833861,PGR,Height (baseline),2.000000e-09,0.010505,[0.0071-0.0139] unit increase,...,rs586143-C,0,586143,intron_variant,0.0,0.839123,8.698970,NaN,Affymetrix [9804479] (imputed),N
213,Alu,chr11:116975263,chr11:116988959:C,rs77852530,0.934272,SIK3,PCSK9 levels in statin-naive individuals,1.000000e-06,0.052470,[0.031-0.073] unit decrease,...,rs77852530-C,0,77852530,intron_variant,0.0,0.0536782472520283,6.000000,NaN,"Affymetrix, Illumina [9095656] (imputed)",N
214,Alu,chr11:127071238,chr11:127067551:T,rs76701,0.971525,KIRREL3-AS3 - LINC02712,Drinks per week,1.000000e-09,0.005370,[0.0036-0.0072] unit increase,...,rs76701-T,0,76701,intron_variant,1.0,0.278,9.000000,NaN,NR [NR] (imputed),N
215,Alu,chr11:131454756,chr11:131455352:T,rs11601906,0.711992,NTM,Obesity-related traits,2.000000e-06,0.040000,[NR] ng/dL increase,...,rs11601906-A,0,11601906,intron_variant,0.0,0.021,5.698970,(Ft4 ),Illumina [899892],N
207,Alu,chr11:32412732,chr11:32402750:T,rs5030241,0.906006,WT1,Inguinal hernia,7.000000e-11,1.080000,NaN,...,rs5030241-A,0,5030241,intron_variant,0.0,0.71,10.154902,NaN,Affymetrix [8568156] (imputed),N
